# Installing & Importing Libraries

In [ ]:
#install stable versions of the needed libraries
!pip uninstall -y numpy scipy scikit-learn xgboost catboost lightgbm shap numba scikeras
!pip install numpy==1.26
!pip install scipy==1.15.2
!pip install scikit-learn==1.5.2
!pip install numba==0.60.0
!pip install shap==0.48.0
!pip install xgboost==2.1.1    # using this stable version fixes error with shap
!pip install lightgbm catboost scikeras
!pip install -q category_encoders

In [ ]:
#importing modules
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import xgboost as xgb
import lightgbm as lgb
import catboost as cat
import tensorflow as tf
import shap
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, roc_auc_score, make_scorer,confusion_matrix
from sklearn.ensemble import RandomForestClassifier, StackingClassifier
from sklearn.neural_network import MLPClassifier
from sklearn.linear_model import LogisticRegression
from catboost import CatBoostClassifier
from xgboost import XGBClassifier
from lightgbm import LGBMClassifier
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, Dropout
from tensorflow.keras.optimizers import Adam
from scikeras.wrappers import KerasClassifier
import category_encoders as ce
from sklearn.pipeline import Pipeline
from imblearn.pipeline import Pipeline as ImbPipeline
from category_encoders import TargetEncoder
from imblearn.over_sampling import SMOTE
from sklearn.model_selection import cross_val_score
from tensorflow.keras.layers import Input
from sklearn.base import BaseEstimator, TransformerMixin
import warnings
warnings.filterwarnings("ignore")

# Loading the Dataset

In [ ]:
# Loading dataset to pandas dataFrame
df = pd.read_csv('/content/sample_data/credit_card_dataset.csv')

#Dataset Cleaning &  Preprocessing

In [ ]:
# Remove the 'fraud_' prefix from the merchant column
df['merchant'] = df['merchant'].str.replace('^fraud_', '', regex=True)

# Check
df['merchant'].head()


,merchant
0,Kirlin and Sons
1,Sporer-Keebler
2,"Swaniawski, Nitzsche and Welch"
3,Haley Group
4,Johnston-Casper


In [ ]:
df['is_fraud'].value_counts()

,count
is_fraud,
0,553574
1,2145


#Feature Engineering

### dropping meaningless features

In [ ]:
drop_cols = ['sn','first','last','street','trans_num','cc_num']   #drop features that add no value to model
df = df.drop(columns=drop_cols)

###To date time

In [ ]:
# To enable ease of deployement, Datetime preprocessing is
# integrated inside model pipeline to avoid errors in  datetime preprocessing functions

class DateTimeTransformer(BaseEstimator, TransformerMixin):
    def __init__(self, datetime_col='trans_date_trans_time', dob_col='dob'):
        self.datetime_col = datetime_col
        self.dob_col = dob_col

    def fit(self, df, y=None):
        return self

    def transform(self, df):
        df = df.copy()

        # Transaction datetime features
        if self.datetime_col in df:
            dt = pd.to_datetime(df[self.datetime_col], dayfirst=True)
            df['Year'], df['Month'], df['Day'], df['Hour'] = dt.dt.year, dt.dt.month, dt.dt.day, dt.dt.hour
            df['IsWeekend'] = (dt.dt.dayofweek >= 5).astype(int)

        # Age from DOB
        if self.dob_col in df and 'Year' in df:
            dob = pd.to_datetime(df[self.dob_col], dayfirst=True)
            df['age'] = df['Year'] - dob.dt.year

        return df

# Splitting the dataset into Train & Test

In [ ]:
#Data is splitted

X = df.drop('is_fraud', axis=1)
y = df['is_fraud']
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

#Encoding of categorical Features and Scaling of Numerical Features

In [ ]:
X_train.nunique()

,0
trans_date_trans_time,208758
merchant,693
category,14
amt,34507
gender,2
city,849
state,50
zip,912
lat,910
long,910


In [ ]:

# Encoding is modified to select specfic features that will be used in streamlit form
# Column is selected Manually to avoid errors.

# Dynamic numeric transformer
class DynamicNumericTransformer(BaseEstimator, TransformerMixin):
    def __init__(self, high_cat=[], low_cat=[]):
        self.high_cat = high_cat
        self.low_cat = low_cat
        self.scaler = StandardScaler()
        self.numeric_cols = []

    def fit(self, df, y=None):
        # select only numeric columns
        self.numeric_cols = [
            c for c in df.columns
            if c not in self.high_cat + self.low_cat and pd.api.types.is_numeric_dtype(df[c])
        ]
        self.scaler.fit(df[self.numeric_cols].fillna(0))
        return self

    def transform(self, df):
        return self.scaler.transform(df[self.numeric_cols].fillna(0))

    def get_feature_names_out(self, input_features=None):
        return self.numeric_cols


# Column lists
numeric_features = ['amt', 'zip', 'lat', 'long', 'city_pop', 'unix_time',
 'merch_lat', 'merch_long', 'Year', 'Month', 'Day', 'Hour', 'IsWeekend', 'age']
high_card_cat = ['merchant', 'city', 'job']
low_card_cat = ['category', 'state', 'gender']


# Preprocessor
preprocessor = ColumnTransformer(
    transformers=[
        ('num', StandardScaler(), numeric_features),
        ('low_cat', OneHotEncoder(handle_unknown='ignore', sparse_output=False), low_card_cat),
        ('high_cat', TargetEncoder(), high_card_cat)
    ],
    remainder='drop'
)

# initialize Evaluation metrics

In [ ]:
#initialing evaluation metrics
def evaluate(y_true, y_pred, y_prob):
    acc, prec, rec, f1, auc = (
        accuracy_score(y_true,y_pred),
        precision_score(y_true,y_pred,zero_division=0),
        recall_score(y_true,y_pred,zero_division=0),
        f1_score(y_true,y_pred,zero_division=0),
        roc_auc_score(y_true,y_prob) if len(set(y_true))>1 else 0
    )
    tn, fp, fn, tp = confusion_matrix(y_true, y_pred).ravel()
    return {'accuracy':acc,'precision':prec,'recall':rec,'f1':f1,'auc':auc,'tn':tn,'fp':fp,'fn':fn,'tp':tp}

# MODELLING

# Initializing classifiers

In [ ]:
#RF,xgboost,catboost,lightboost and MLP are initialized and reused via a pipeline
import random
rf = RandomForestClassifier(             #  ------------------ random_forest
    n_estimators=100,
    random_state=42,
    class_weight="balanced",
    n_jobs=1
)

# compute scale_pos_weight
scale_pos_weight = (len(y_train) - sum(y_train)) / sum(y_train) #  ------------------ xg_Boost
xgbc = XGBClassifier(
    n_estimators=100,
    n_jobs=1,
    random_state=42,
    scale_pos_weight=scale_pos_weight,
    use_label_encoder=False,
    eval_metric='logloss'
)

cat_clf = CatBoostClassifier(              #  ------------------ cat_Boost
    iterations=100,
    random_state=42,
    auto_class_weights="Balanced",
    #thread_count=1,
    verbose=0
)

lgb_clf = lgb.LGBMClassifier(            #  -------------------- Light_Boost
    n_estimators=100,
    random_state=42,
    n_jobs=1,
    verbose=0,
    class_weight='balanced'
)

#  Seeds to fixruntime error
tf.random.set_seed(42)
np.random.seed(42)
random.seed(42)

#  MLP
def build_mlp(meta):
    n_features = meta["n_features_in_"]

    model = Sequential([
        Input(shape=(n_features,)),
        Dense(64, activation='relu'),
        Dropout(0.2),
        Dense(32, activation='relu'),
        Dropout(0.2),
        Dense(1, activation='sigmoid')
    ])

    model.compile(
        optimizer=Adam(learning_rate=0.001),
        loss='binary_crossentropy',
        metrics=['accuracy']
    )
    return model


#  SAFE WRAPPER ( for 2 dimension  output (n_samples, 2)) this is important to fix single sigmoid output error
class SafeKerasClassifier(KerasClassifier):
    def predict_proba(self, X, **kwargs):
        proba = super().predict_proba(X, **kwargs)

        if proba.ndim == 1:
            proba = proba.reshape(1, -1)
        if proba.shape[1] == 1:
            proba = np.hstack([1 - proba, proba])

        return proba


# Final  keras classifier
mlp_clf = SafeKerasClassifier(
    model=build_mlp,
    epochs=1,     #----------------------Due to computational overhead this was intentionally reduced from 20 to 1
    batch_size=32,
    verbose=0,
    validation_split=0.1,
    random_state=42
)

# HYBRID MODEL STACKING

### Initializing Stack

In [ ]:
stack_model = StackingClassifier(   # Hybrid model: stacking all base-models using LR meta-learner
    estimators=[
        ('rf', rf),
        ('xgb', xgbc),
        ('cat', cat_clf),
        ('lgbm', lgb_clf),
        ('mlp', mlp_clf)
    ],
    final_estimator=LogisticRegression(),
    passthrough=True,   #------------------- This is important for shap: It allows the initial features to be visible through the stack
    stack_method='predict_proba',
    n_jobs=1
)

## HYBRID STACK + SMOTE

In [ ]:
# Hybrid Model stack pipeline
pipeline_with_smote = ImbPipeline([
    ('feature_engineering', DateTimeTransformer(datetime_col='trans_date_trans_time', dob_col='dob')),
    ('preprocess', preprocessor),
    ('smote', SMOTE(random_state=42)),
    ('stack', stack_model)
])

pipeline_with_smote.fit(X_train, y_train)



,steps,"[('feature_engineering', ...), ('preprocess', ...), ...]"
,transform_input,None
,memory,None
,verbose,False
,datetime_col,'trans_date_trans_time'
,dob_col,'dob'
,"transformers transformers: list of tuplesList of (name, transformer, columns) tuples specifying thetransformer objects to be applied to subsets of the data.name : str Like in Pipeline and FeatureUnion, this allows the transformer and its parameters to be set using ``set_params`` and searched in grid search.transformer : {'drop', 'passthrough'} or estimator Estimator must support :term:`fit` and :term:`transform`. Special-cased strings 'drop' and 'passthrough' are accepted as well, to indicate to drop the columns or to pass them through untransformed, respectively.columns : str, array-like of str, int, array-like of int, array-like of bool, slice or callable Indexes the data on its second axis. Integers are interpreted as positional columns, while strings can reference DataFrame columns by name. A scalar string or int should be used where ``transformer`` expects X to be a 1d array-like (vector), otherwise a 2d array will be passed to the transformer. A callable is passed the input data `X` and can return any of the above. To select multiple columns by name or dtype, you can use :obj:`make_column_selector`.","[('num', ...), ('low_cat', ...), ...]"
,"remainder remainder: {'drop', 'passthrough'} or estimator, default='drop'By default, only the specified columns in `transformers` aretransformed and combined in the output, and the non-specifiedcolumns are dropped. (default of ``'drop'``).By specifying ``remainder='passthrough'``, all remaining columns thatwere not specified in `transformers`, but present in the data passedto `fit` will be automatically passed through. This subset of columnsis concatenated with the output of the transformers. For dataframes,extra columns not seen during `fit` will be excluded from the outputof `transform`.By setting ``remainder`` to be an estimator, the remainingnon-specified columns will use the ``remainder`` estimator. Theestimator must support :term:`fit` and :term:`transform`.Note that using this feature requires that the DataFrame columnsinput at :term:`fit` and :term:`transform` have identical order.",'drop'
,"sparse_threshold sparse_threshold: float, default=0.3If the output of the different transformers contains sparse matrices,these will be stacked as a sparse matrix if the overall density islower than this value. Use ``sparse_threshold=0`` to always returndense. When the transformed output consists of all dense data, thestacked result will be dense, and this keyword will be ignored.",0.3
,"n_jobs n_jobs: int, default=NoneNumber of jobs to run in parallel.``None`` means 1 unless in a :obj:`joblib.parallel_backend` context.``-1`` means using all processors. See :term:`Glossary `for more details.",None
,"transformer_weights transformer_weights: dict, default=NoneMultiplicative weights for features per transformer. The output of thetransformer is multiplied by these weights. Keys are transformer names,values the weights.",None


In [ ]:
# Predictions
stack_pred_smote = pipeline_with_smote.predict(X_test)

# Probabilities (needed for ROC, AUC, PR, etc.)
stack_prob_smote = pipeline_with_smote.predict_proba(X_test)[:, 1]


In [ ]:
print("Accuracy :", accuracy_score(y_test, stack_pred_smote))
print("Precision:", precision_score(y_test, stack_pred_smote))
print("Recall   :", recall_score(y_test, stack_pred_smote))
print("F1-score :", f1_score(y_test, stack_pred_smote))
print("ROC-AUC  :", roc_auc_score(y_test, stack_prob_smote))
print("Confusion Matrix:\n", confusion_matrix(y_test, stack_pred_smote))

Accuracy : 0.9988483408910962
Precision: 0.8661800486618005
Recall   : 0.8298368298368298
F1-score : 0.8476190476190476
ROC-AUC  : 0.977296607861572
Confusion Matrix:
 [[110660     55]
 [    73    356]]


Accuracy : 0.9988483408910962
Precision: 0.8661800486618005
Recall   : 0.8298368298368298
F1-score : 0.8476190476190476
ROC-AUC  : 0.977296607861572
Confusion Matrix:
 [[110660     55]
 [    73    356]]


###Exporting Model

In [ ]:
import joblib
# Save the trained pipeline
joblib.dump(pipeline_with_smote, "Hybrid_Stack_on_SMOTE2Dfix.pkl")


['Hybrid_Stack_on_SMOTE2Dfix.pkl']

In [ ]:
from google.colab import files

# Downoad model to local enviroment
files.download('Hybrid_Stack_on_SMOTE2Dfix.pkl')

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

# Improving Interpretability with SHAP

In [ ]:
#importing model into environment
import joblib
model= joblib.load("/content/Hybrid_Stack_on_SMOTE2Dfix.pkl")

In [ ]:
print("Full pipeline structure:")
for name, step in model.named_steps.items():
    print(f"{name:20s} : {type(step).__name__}")

Full pipeline structure:
feature_engineering  : DateTimeTransformer
preprocess           : ColumnTransformer
smote                : SMOTE
stack                : StackingClassifier


In [ ]:
X_sample = X_test.iloc[[69669]]   # A random sample from test data

X_fe  = model.named_steps['feature_engineering'].transform(X_sample) #- recreating prediction process excluding SMOTE
X_pre = model.named_steps['preprocess'].transform(X_fe)
stack_clf = model.named_steps['stack']

print("Testing individual base on transformed input (X_pre.shape =", X_pre.shape, ")")
print("-" * 70)

for i, est in enumerate(stack_clf.estimators_):
    try:
        proba = est.predict_proba(X_pre)
        print(f"Base {i}: {type(est).__name__:<18} → proba shape = {proba.shape}")  # get output shape
    except Exception as e:
        print(f"Base {i}: {type(est).__name__:<18} → FAILED: {str(e)[:100]}...")

Testing individual base on transformed input (X_pre.shape = (1, 83) )
----------------------------------------------------------------------
Base 0: RandomForestClassifier → proba shape = (1, 2)
Base 1: XGBClassifier      → proba shape = (1, 2)
Base 2: CatBoostClassifier → proba shape = (1, 2)
Base 3: LGBMClassifier     → proba shape = (1, 2)
Base 4: SafeKerasClassifier → proba shape = (1, 2)


In [ ]:
import shap
import pandas as pd
import numpy as np
#Shap Module does not support stacking classifier, so we aggregate each model shap values
#Testing shap on a single base model.
#  Sample + prediction
X_sample = X_test.iloc[[69669]]   #selecting a random sample from test data

prediction = model.predict(X_sample)[0]   #predict
pred_prob  = model.predict_proba(X_sample)[0, 1]   #prediction probability

print(f"Prediction (fraud=1): {prediction}")
print(f"Fraud probability: {pred_prob:.4f}\n")


# Transform data
X_fe = model.named_steps['feature_engineering'].transform(X_sample)
X_transformed = model.named_steps['preprocess'].transform(X_fe)

# 3. Use RandomForest (index 0)
stack_clf = model.named_steps['stack']
rf_model = stack_clf.estimators_[0]          #from our stack first basemodel is Rf in index[0]

print(f"Explaining: {type(rf_model).__name__}")

#  SHAP

explainer = shap.TreeExplainer(rf_model)
shap_values_all = explainer.shap_values(X_transformed)

# Binary → class 1 (fraud)
shap_values = shap_values_all[1][0] if isinstance(shap_values_all, list) else shap_values_all[0]

#  grouping logic
#All transformed features are remapped to its
# original column using column name tag.
feature_names_transformed = model.named_steps['preprocess'].get_feature_names_out()
numeric_features = ['amt', 'zip', 'lat', 'long', 'city_pop', 'unix_time',
                    'merch_lat', 'merch_long', 'Year', 'Month', 'Day', 'Hour', 'IsWeekend', 'age']
low_card_cat = ['category', 'state', 'gender']
high_card_cat = ['merchant', 'city', 'job']

all_features = numeric_features + low_card_cat + high_card_cat

contrib_dict = {}
for feature in all_features:
    indices = [i for i, name in enumerate(feature_names_transformed) if feature in name]
    if indices:
        contrib_dict[feature] = shap_values[indices].sum()


# Final table
# and shapvalues
contrib_df = pd.DataFrame(list(contrib_dict.items()), columns=['Feature', 'SHAP_Contribution'])

total_abs = np.abs(contrib_df['SHAP_Contribution']).sum()
contrib_df['Contribution_pct'] = (np.abs(contrib_df['SHAP_Contribution']) / total_abs * 100).round(2)

contrib_df = contrib_df.sort_values(by='SHAP_Contribution', key=abs, ascending=False)

print(contrib_df.round(4))

Prediction (fraud=1): 0
Fraud probability: 0.0000

Explaining: RandomForestClassifier
       Feature  SHAP_Contribution  Contribution_pct
18        city                0.0             17.17
14    category                0.0             12.37
17    merchant                0.0             11.26
7   merch_long                0.0             10.40
3         long                0.0             10.33
6    merch_lat                0.0              9.01
2          lat                0.0              6.40
1          zip               -0.0              5.58
13         age               -0.0              5.06
4     city_pop                0.0              4.68
9        Month               -0.0              1.88
15       state                0.0              1.82
19         job                0.0              1.08
5    unix_time               -0.0              0.83
10         Day               -0.0              0.73
11        Hour               -0.0              0.70
0          amt                

## Aggregating individual model explanability

In [ ]:
import shap
import pandas as pd
import numpy as np
from shap.utils import safe_isinstance

#
# random sample
X_sample = X_test.iloc[[6969]]

pred = model.predict(X_sample)[0]
prob = model.predict_proba(X_sample)[0, 1]

print(f"Prediction (fraud=1): {pred}")
print(f"Fraud probability: {prob:.6f}\n")


X_fe = model.named_steps['feature_engineering'].transform(X_sample) #sample transform
X_pre = model.named_steps['preprocess'].transform(X_fe)             #sample preprocessing

#  grouping logic
#All transformed features are remapped to its
# original column using column name tag
feature_names = model.named_steps['preprocess'].get_feature_names_out()

numeric_features = ['amt', 'zip', 'lat', 'long', 'city_pop', 'unix_time',
                    'merch_lat', 'merch_long', 'Year', 'Month', 'Day', 'Hour', 'IsWeekend', 'age']
low_card_cat = ['category', 'state', 'gender']
high_card_cat = ['merchant', 'city', 'job']
groupable_features = numeric_features + low_card_cat + high_card_cat


# Model list (all 5)
stack_clf = model.named_steps['stack']
model_info = [
    ("RandomForest", stack_clf.estimators_[0]),
    ("XGBoost",      stack_clf.estimators_[1]),
    ("CatBoost",     stack_clf.estimators_[2]),
    ("LightGBM",     stack_clf.estimators_[3]),
    ("Keras",        stack_clf.estimators_[4]),
]

# SHAP computation per model
all_contributions = {f: [] for f in groupable_features}
successful = []

for name, model in model_info:
    print(f"→ {name:12} ", end="")
    try:
        if name == "Keras":
            # KernelExplainer for neural nets as keras does not support Tree explainer
            explainer = shap.KernelExplainer(model.predict_proba, X_pre)
            shap_all = explainer.shap_values(X_pre)
            shap_vals = shap_all[1][0] if isinstance(shap_all, list) else shap_all[0]

        else:
            # TreeExplainer for tree models
            explainer = shap.TreeExplainer(
                model,
                model_output="raw",
                feature_perturbation="tree_path_dependent")

            # XGBoost workaround: force base_score to float
            if name == "XGBoost":
                  try:
                      booster = model.get_booster()
                      # Get current base_score (may be string)
                      current_bs = booster.attributes().get('base_score', '0.5')
                      if isinstance(current_bs, str):
                          # Clean it up
                          cleaned = current_bs.strip('[]')
                          try:
                              float_bs = float(cleaned)
                              booster.set_attr('base_score', str(float_bs))
                              print("   → patched base_score string → float")
                          except ValueError:
                              print("   → could not parse base_score")
                  except Exception as patch_err:
                      print(f"   → patch failed: {patch_err}")


            shap_all = explainer.shap_values(X_pre)
            shap_vals = shap_all[1][0] if isinstance(shap_all, list) else shap_all[0]

        # Group # original column using column name tag.
        for feat in groupable_features:
            indices = [i for i, col in enumerate(feature_names) if feat in col]
            if indices:
                contrib = shap_vals[indices].sum()
                all_contributions[feat].append(contrib)

        successful.append(name)
        print("OK")

    except Exception as e:
        print(f"FAILED ({type(e).__name__}: {str(e)[:60]}...)")

# Aggregate
if not successful:
    print("\nNo successful explanations.")
else:
    agg_rows = []
    for feat, vals in all_contributions.items():
        if vals:
            mean_v = np.mean(vals)
            std_v  = np.std(vals) if len(vals) > 1 else 0.0
            agg_rows.append({
                'Feature'         : feat,
                'Mean_SHAP'       : mean_v,
                'Std_SHAP'        : std_v,
                'Num_Models'      : len(vals),
            })

    agg_df = pd.DataFrame(agg_rows)   # Return values in dataFrame
    if not agg_df.empty:
        total_abs = np.abs(agg_df['Mean_SHAP']).sum()
        if total_abs > 0:
            agg_df['Contribution_pct'] = (np.abs(agg_df['Mean_SHAP']) / total_abs * 100).round(2)
        else:
            agg_df['Contribution_pct'] = 0.0

        agg_df = agg_df.sort_values('Mean_SHAP', key=abs, ascending=False).reset_index(drop=True)

        print("\n" + "═"*75)
        print("COMBINED FEATURE-LEVEL SHAP ACROSS ALL SUCCESSFUL BASE MODELS")
        print("═"*75)
        print(agg_df.round(4))
    else:
        print("No feature contributions collected.")

Prediction (fraud=1): 0
Fraud probability: 0.000024

→ RandomForest OK
→ XGBoost         → patch failed: Booster.set_attr() takes 1 positional argument but 3 were given
OK
→ CatBoost     OK
→ LightGBM     OK
→ Keras        

  0%|          | 0/1 [00:00<?, ?it/s]

OK

═══════════════════════════════════════════════════════════════════════════
COMBINED FEATURE-LEVEL SHAP ACROSS ALL SUCCESSFUL BASE MODELS
═══════════════════════════════════════════════════════════════════════════
       Feature  Mean_SHAP  Std_SHAP  Num_Models  Contribution_pct
0         city    -3.8178    3.3745           5             39.14
1          amt    -2.0184    1.9365           5             20.69
2     category    -1.3632    1.7172           5             13.97
3         Hour    -0.7861    0.7579           5              8.06
4     merchant    -0.4321    0.4680           5              4.43
5          age    -0.2748    0.3572           5              2.82
6        Month    -0.2571    0.2966           5              2.64
7          zip    -0.2085    0.3231           5              2.14
8          Day     0.1529    0.1414           5              1.57
9    unix_time    -0.1197    0.2589           5              1.23
10         lat    -0.0897    0.1940           5         

In [ ]:
!pip freeze > requirements.txt